# Experimentaion NOTEBOOK
In this notebook we will try different baseline models with related preprocessing pieline. We will try to build a baseline preprocessing pipeline and then we will try to improve it by trying different techniques and see how it affects the model performance. 

Preprocessing is model dependent, so we will try differnt pipeline accordingly and test it with different models. We will try to find the best preprocessing pipeline for our dataset and then we will use it for feature engineering and model tuning in next notebooks.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/Indian_bank_churn_orig.csv')
df.head()

,CustomerID,Age,Gender,City,Employment_Type,Annual_Income,Credit_Score,Tenure_Months,Avg_Monthly_Balance,Balance_Change_Ratio,Mobile_App_Logins,UPI_Transactions,ATM_Withdrawals,NetBanking_Usage,Complaint_Tickets,Call_Center_Interactions,Has_Credit_Card,Has_Loan,Has_Insurance,Churn
0,IB000001,54,Male,Rajkot,Business,1130100.0,684.0,163,110960.0,-0.0256,5.0,10.0,NaN,7.0,1.0,3.0,1,0,1,0
1,IB000002,28,Female,Jaipur,Salaried,472700.0,626.0,103,32200.0,0.0649,19.0,18.0,2.0,8.0,0.0,5.0,1,0,0,0
2,IB000003,35,Female,Bengaluru,Salaried,742800.0,637.0,126,42100.0,0.0616,6.0,12.0,NaN,6.0,0.0,2.0,1,0,0,0
3,IB000004,37,Female,Vadodara,Student,273000.0,550.0,91,8820.0,-0.0372,9.0,8.0,2.0,9.0,1.0,1.0,0,0,0,0
4,IB000005,26,Male,Pune,Salaried,1294000.0,590.0,67,137960.0,-0.1253,12.0,26.0,5.0,17.0,0.0,1.0,0,0,0,0


# Save train and test files for further use

In [2]:
from sklearn.model_selection import train_test_split
X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,   # stratify Because churn is imbalanced, don’t want distribution mismatch.
    random_state=42
)

train_df = X_train.copy()
train_df["Churn"] = y_train

test_df = X_test.copy()
test_df["Churn"] = y_test

train_df.to_csv("../data/processed/train.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)

## Now start actual preprocessing work

In [3]:
# From eda we know numeric and categorical features
numeric_features = ['Age', 'Annual_Income', 'Credit_Score', 'Tenure_Months',
       'Avg_Monthly_Balance', 'Balance_Change_Ratio', 'Mobile_App_Logins',
       'UPI_Transactions', 'ATM_Withdrawals', 'NetBanking_Usage',
       'Complaint_Tickets', 'Call_Center_Interactions', 'Has_Credit_Card',
       'Has_Loan', 'Has_Insurance'] # Dont add target column here
categorical_features = ['CustomerID', 'Gender', 'City', 'Employment_Type']

### Before trying best, Build baseline Preprocessing Pipeline

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

In [5]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)), # Add indicator to capture missingness pattern , this can be useful for some models to learn from the fact that a value was missing.
    ("scaler", StandardScaler())
])

In [6]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [7]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [8]:
preprocessor.fit(X_train)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(add_indicator=True,
                                                                strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Age', 'Annual_Income', 'Credit_Score',
                                  'Tenure_Months', 'Avg_Monthly_Balance',
                                  'Balance_Change_Ratio', 'Mobile_App_Logins',
                                  'UPI_Transactions', 'ATM_Withdrawals',
                                  'NetBanking_Usage', 'Complaint_Tickets',
                                  'Call_Center_Interactions', 'Has_Credit_Card',
                                  'Has_Loan', 'Has_Insurance']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['CustomerID', 'Gender', 'City',
                                  'Employment_Type'])])

In [9]:
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [10]:
print("Shape of X_train_transformed:", X_train_transformed.shape)
print("Shape of X_test_transformed:", X_test_transformed.shape)

Shape of X_train_transformed: (40000, 40072)
Shape of X_test_transformed: (10000, 40072)


### Try a Quick Baseline Model

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression())
])

baseline_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(add_indicator=True,
                                                                                 strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Annual_Income',
                                                   'Credit_Score',
                                                   'Tenure_Months',
                                                   'Avg_Monthly_Balance',
                                                   'Balance_Change_Ratio',
                                                   'Mobile_App_Logins',
                                                   'UPI_Transactions',
                                                   'ATM_Withdrawals',
                                                   'NetBanking_Usage',
                                                   'Complaint_Tickets',
                                                   'Call_Center_Interactions',
                                                   'Has_Credit_Card',
                                                   'Has_Loan',
                                                   'Has_Insurance']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['CustomerID', 'Gender',
                                                   'City',
                                                   'Employment_Type'])])),
                ('model', LogisticRegression())])

In [12]:
from sklearn.metrics import roc_auc_score

y_pred = baseline_model.predict_proba(X_test)[:,1]
roc_auc_score(y_test, y_pred)

np.float64(0.7728532677078578)

### Define one function for evaluation so that we can use it for all models and pipelines

In [13]:
def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_test)[:,1]
    return roc_auc_score(y_test, y_pred)

In [14]:
print("Results of Baseline Model (Logistic Regression):")
print("Baseline pipeline:  meadian imputation, standard scaling, one hot encoding")
evaluate_model(baseline_model, X_train, y_train, X_test, y_test)

Results of Baseline Model (Logistic Regression):
Baseline pipeline:  meadian imputation, standard scaling, one hot encoding


np.float64(0.7728532677078578)

## Preprocessing quality is model-dependent.
Example:
- Scaling matters for Logistic Regression.
- Scaling does NOT matter for RandomForest.
- OneHot can explode dimensionality and hurt some models.
- SMOTE interacts differently with linear vs tree models.

2. Lets try same logistic regresion with class weight and see if it improves the performance.

In [15]:
from sklearn.linear_model import LogisticRegression
logistic_model_with_class_weight = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(class_weight="balanced"))
])
logistic_model_with_class_weight.fit(X_train, y_train)

C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(add_indicator=True,
                                                                                 strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Annual_Income',
                                                   'Credit_Score',
                                                   'Tenure_Months',
                                                   'Avg_Monthly_Balance',
                                                   'Balance_Change_Ratio',
                                                   'Mobile_App_Logins',
                                                   'UPI_Transactions',
                                                   'ATM_Withdrawals',
                                                   'NetBanking_Usage',
                                                   'Complaint_Tickets',
                                                   'Call_Center_Interactions',
                                                   'Has_Credit_Card',
                                                   'Has_Loan',
                                                   'Has_Insurance']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['CustomerID', 'Gender',
                                                   'City',
                                                   'Employment_Type'])])),
                ('model', LogisticRegression(class_weight='balanced'))])

In [16]:
evaluate_model(logistic_model_with_class_weight, X_train, y_train, X_test, y_test)


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


np.float64(0.7724861028915357)

3. To handle imbalnce, lets try XGBoost + scale_pos_weight

In [17]:
# To handle imbalnce, lets try XGBoost + scale_pos_weight
from xgboost import XGBClassifier
xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(scale_pos_weight=10))
])
xgb_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(add_indicator=True,
                                                                                 strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Annual_Income',
                                                   'Credit_Score',
                                                   'Tenure_Months',
                                                   'Avg_Monthly_Balance',
                                                   'Balance_Change_Ratio',
                                                   'Mobile_App_Logins',
                                                   'UPI_Transactions',
                                                   'ATM_Withdrawals',
                                                   'NetBanking_U...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [18]:
evaluate_model(xgb_model, X_train, y_train, X_test, y_test)

np.float64(0.774393742439678)

4. lets try random forest with class weight

In [19]:
# Lets design preprocessing pipeline without scaling
preprocessor_without_scaling = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])


In [20]:
# lets try random forest with class weight
from sklearn.ensemble import RandomForestClassifier
rf_model = Pipeline([
    ("preprocessor", preprocessor_without_scaling),
    ("model", RandomForestClassifier(class_weight="balanced"))
])
rf_model.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  SimpleImputer(strategy='median'),
                                                  ['Age', 'Annual_Income',
                                                   'Credit_Score',
                                                   'Tenure_Months',
                                                   'Avg_Monthly_Balance',
                                                   'Balance_Change_Ratio',
                                                   'Mobile_App_Logins',
                                                   'UPI_Transactions',
                                                   'ATM_Withdrawals',
                                                   'NetBanking_Usage',
                                                   'Complaint_Tickets',
                                                   'Call_Center_Interactions',
                                                   'Has_Credit_Card',
                                                   'Has_Loan',
                                                   'Has_Insurance']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['CustomerID', 'Gender',
                                                   'City',
                                                   'Employment_Type'])])),
                ('model', RandomForestClassifier(class_weight='balanced'))])

In [21]:
evaluate_model(rf_model, X_train, y_train, X_test, y_test)

np.float64(0.7636378751091202)

5. Lets try LightBGM and stop

In [25]:
# 5. Lets try LightBGM and decide final prerocessing pipeline and models fr futrther use
from lightgbm import LGBMClassifier
lgbm_model = Pipeline([ 
    ("preprocessor", preprocessor_without_scaling),
    ("model", LGBMClassifier(class_weight="balanced"))
])
lgbm_model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007355 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 62
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  SimpleImputer(strategy='median'),
                                                  ['Age', 'Annual_Income',
                                                   'Credit_Score',
                                                   'Tenure_Months',
                                                   'Avg_Monthly_Balance',
                                                   'Balance_Change_Ratio',
                                                   'Mobile_App_Logins',
                                                   'UPI_Transactions',
                                                   'ATM_Withdrawals',
                                                   'NetBanking_Usage',
                                                   'Complaint_Tickets',
                                                   'Call_Center_Interactions',
                                                   'Has_Credit_Card',
                                                   'Has_Loan',
                                                   'Has_Insurance']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['CustomerID', 'Gender',
                                                   'City',
                                                   'Employment_Type'])])),
                ('model', LGBMClassifier(class_weight='balanced'))])

In [26]:
evaluate_model(lgbm_model, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008631 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 62
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


np.float64(0.7817554004779598)

Results: (ROC AUC Score)
1. Baseline Logistic Regression: 0.7728
2. Logistic Regression with class weight: 0.7724
3. XGBoost with scale_pos_weight: 0.7743
4. Random Forest with class weight: 0.7627
5. LightGBM with class weight: 0.7817

# Final preprocessing pipeline with LightBGM Model

In [9]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)), # Add indicator to capture missingness pattern , this can be useful for some models to learn from the fact that a value was missing.
# no need of scaling for tree based models
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
final_preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])


In [14]:
from sklearn.metrics import roc_auc_score

def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_test)[:,1]
    return roc_auc_score(y_test, y_pred)

In [12]:
from lightgbm import LGBMClassifier
lightBGM = Pipeline([ 
    ("preprocessor", final_preprocessor),
    ("model", LGBMClassifier(class_weight="balanced"))
])
lgbm_model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.554830 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(add_indicator=True,
                                                                                 strategy='median'))]),
                                                  ['Age', 'Annual_Income',
                                                   'Credit_Score',
                                                   'Tenure_Months',
                                                   'Avg_Monthly_Balance',
                                                   'Balance_Change_Ratio',
                                                   'Mobile_App_Logins',
                                                   'UPI_Transactions',
                                                   'ATM_Withdrawals',
                                                   'NetBanking_Usage',
                                                   'Complaint_Tickets',
                                                   'Call_Center_Interactions',
                                                   'Has_Credit_Card',
                                                   'Has_Loan',
                                                   'Has_Insurance']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['CustomerID', 'Gender',
                                                   'City',
                                                   'Employment_Type'])])),
                ('model', LGBMClassifier(class_weight='balanced'))])

In [15]:
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012844 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


np.float64(0.7802335651398216)

In [16]:
print("Finalized baseline accuracy with lightBGM, One Hot Encoding, SimpleImuter: 0.7802 ")

Finalized baseline accuracy with lightBGM, One Hot Encoding, SimpleImuter: 0.7802 


### To handling missing values

In [22]:
(df.isnull().sum() / len(df)) * 100

CustomerID                   0.000
Age                          0.000
Gender                       0.000
City                         0.000
Employment_Type              0.000
Annual_Income               17.260
Credit_Score                 4.762
Tenure_Months                0.000
Avg_Monthly_Balance         12.016
Balance_Change_Ratio        12.016
Mobile_App_Logins           12.562
UPI_Transactions            12.562
ATM_Withdrawals             13.942
NetBanking_Usage            12.562
Complaint_Tickets            4.782
Call_Center_Interactions    14.100
Has_Credit_Card              0.000
Has_Loan                     0.000
Has_Insurance                0.000
Churn                        0.000
dtype: float64

Missing values are less than 10-15%, so mean/meadian imputation is fine.
Advanced imputation techniques like KNN or MICE are not necessary here and may not provide significant benefits given the low percentage of missing data.

### Encoding Decision

In [23]:
for col in categorical_features:
    print(col, df[col].nunique())

CustomerID 50000
Gender 2
City 40
Employment_Type 5


So for encoding, we will go with one hot encoding for low cardinality features(Gender, Emloyment tye, churn.... customerID not part of data while training)  
Frequency/ target encoding for city.
 

## Outlier Detection and Handling

Conclusion:

Among all tested models and preprocessing techniques,
LightGBM with class_weight and One-Hot Encoding achieved the best ROC-AUC score of 0.7802.

This configuration will be used as the baseline model for further experimentation.
Future improvements will focus on:
- Advanced feature engineering
- Hyperparameter tuning
- Threshold optimization
- Calibration
- Ensembling